# <span style="font-width:bold; font-size: 3rem; color:#1EB182;"> **Air Quality** </span><span style="font-width:bold; font-size: 3rem; color:#333;">- Part 04: Batch Inference</span>

## 🗒️ This notebook is divided into the following sections:

1. Download model and batch inference data
2. Make predictions, generate PNG for forecast
3. Store predictions in a monitoring feature group and generate PNG for hindcast

## <span style='color:#ff5f27'> 📝 Imports

In [ ]:
import sys
from pathlib import Path

root_dir = Path().absolute()
# Strip ~/notebooks/ccfraud from PYTHON_PATH if notebook started in one of these subdirectories
if root_dir.parts[-1:] == ('notebooks',):
    root_dir = Path(*root_dir.parts[:-1])
    sys.path.append(str(root_dir))
if root_dir.parts[-1:] == ('airquality',):
    root_dir = Path(*root_dir.parts[:-1])
    sys.path.append(str(root_dir))
root_dir = str(root_dir) 

print(f"Root dir: {root_dir}")

# Set the environment variables from the file <root_dir>/.env
from mlfs import config
settings = config.HopsworksSettings(_env_file=f"{root_dir}/.env")

In [ ]:
import datetime
import pandas as pd
from xgboost import XGBRegressor
import hopsworks
import json
from airquality import util

In [ ]:
today = datetime.datetime.now() - datetime.timedelta(0)
tomorrow = today + datetime.timedelta(days = 1)
today

## <span style="color:#ff5f27;"> 📡 Connect to Hopsworks Feature Store </span>

In [ ]:
project = hopsworks.login()
fs = project.get_feature_store() 

secrets = hopsworks.get_secrets_api()
location_str = secrets.get_secret("SENSOR_LOCATION_JSON").value
location = json.loads(location_str)
country=location['country']
city=location['city']
street=location['street']

## <span style="color:#ff5f27;">🪝 Download the model from Model Registry</span>

In [ ]:
mr = project.get_model_registry()

retrieved_model = mr.get_model(
    name="air_quality_xgboost_model",
    version=1,
)

fv = retrieved_model.get_feature_view()

# Download the saved model artifacts to a local directory
saved_model_dir = retrieved_model.download()

In [ ]:
# Loading the XGBoost regressor model and label encoder from the saved model directory
# retrieved_xgboost_model = joblib.load(saved_model_dir + "/xgboost_regressor.pkl")
retrieved_xgboost_model = XGBRegressor()

retrieved_xgboost_model.load_model(saved_model_dir + "/model.json")

# Displaying the retrieved XGBoost regressor model
retrieved_xgboost_model

## <span style="color:#ff5f27;">✨ Get Weather Forecast Features with Feature View   </span>



In [ ]:
weather_fg = fs.get_feature_group(
    name='weather',
    version=1,
)
batch_data = weather_fg.filter(weather_fg.date >= today).read()
batch_data

### <span style="color:#ff5f27;">🤖 Making the predictions</span>

In [ ]:
batch_data['predicted_pm25'] = retrieved_xgboost_model.predict(
    batch_data[['temperature_2m_mean', 'precipitation_sum', 'wind_speed_10m_max', 'wind_direction_10m_dominant']])
batch_data

In [ ]:
batch_data.info()

### <span style="color:#ff5f27;">🤖 Saving the predictions (for monitoring) to a Feature Group</span>

In [ ]:
batch_data['street'] = street
batch_data['city'] = city
batch_data['country'] = country
# Fill in the number of days before the date on which you made the forecast (base_date)
batch_data['days_before_forecast_day'] = range(1, len(batch_data)+1)
batch_data = batch_data.sort_values(by=['date'])
batch_data

In [ ]:
batch_data.info()

### Create Forecast Graph
Draw a graph of the predictions with dates as a PNG and save it to the github repo
Show it on github pages

In [ ]:

pred_file_path = f"{root_dir}/docs/air-quality/assets/img/pm25_forecast.png"
plt = util.plot_air_quality_forecast(city, street, batch_data, pred_file_path)

plt.show()

In [ ]:
# Get or create feature group
monitor_fg = fs.get_or_create_feature_group(
    name='aq_predictions',
    description='Air Quality prediction monitoring',
    version=1,
    primary_key=['city','street','date','days_before_forecast_day'],
    event_time="date"
)

In [ ]:
monitor_fg.insert(batch_data, wait=True)

In [ ]:
# We will create a hindcast chart for  only the forecasts made 1 day beforehand
monitoring_df = monitor_fg.filter(monitor_fg.days_before_forecast_day == 1).read()
monitoring_df

In [ ]:
air_quality_fg = fs.get_feature_group(name='air_quality', version=1)
air_quality_df = air_quality_fg.read()
air_quality_df

In [ ]:
outcome_df = air_quality_df[['date', 'pm25']]
outcome_df['date'] = outcome_df['date'].dt.tz_localize(None)
preds_df =  monitoring_df[['date', 'predicted_pm25']]

hindcast_df = pd.merge(preds_df, outcome_df, on="date")
hindcast_df = hindcast_df.sort_values(by=['date'])

# If there are no outcomes for predictions yet, generate some predictions/outcomes from existing data
if len(hindcast_df) == 0:
    hindcast_df = util.backfill_predictions_for_monitoring(weather_fg, air_quality_df, monitor_fg, retrieved_xgboost_model)
hindcast_df

### Plot the Hindcast comparing predicted with forecasted values (1-day prior forecast)

__This graph will be empty to begin with - this is normal.__

After a few days of predictions and observations, you will get data points in this graph.

In [ ]:
hindcast_file_path = f"{root_dir}/docs/air-quality/assets/img/pm25_hindcast_1day.png"
plt = util.plot_air_quality_forecast(city, street, hindcast_df, hindcast_file_path, hindcast=True)
plt.show()

### Upload the prediction and hindcast dashboards (png files) to Hopsworks


In [ ]:
dataset_api = project.get_dataset_api()
str_today = today.strftime("%Y-%m-%d")
if dataset_api.exists("Resources/airquality") == False:
    dataset_api.mkdir("Resources/airquality")
dataset_api.upload(pred_file_path, f"Resources/airquality/{city}_{street}_{str_today}", overwrite=True)
dataset_api.upload(hindcast_file_path, f"Resources/airquality/{city}_{street}_{str_today}", overwrite=True)

proj_url = project.get_url()
print(f"See images in Hopsworks here: {proj_url}/settings/fb/path/Resources/airquality")

### Upload training hindcast from model artifacts to Resources

In [ ]:
import os

training_hindcast_path = os.path.join(saved_model_dir, "images", "pm25_hindcast.png")
if os.path.exists(training_hindcast_path):
    dataset_api.upload(training_hindcast_path, "Resources/airquality", overwrite=True)
    print(f"Uploaded training hindcast to Resources/airquality/pm25_hindcast.png")
else:
    print(f"Warning: Training hindcast not found at {training_hindcast_path}")

### Create Hopsworks Dashboard with forecast and hindcast charts

In [ ]:
# import subprocess
# import json as json_mod
# import os

# CHART_DIR = "/hopsfs/Resources/charts/airquality"
# os.makedirs(CHART_DIR, exist_ok=True)

# proj_url = project.get_url()
# from hopsworks_common import client as hw_client
# _client = hw_client.get_instance()
# project_id = _client._project_id

# # --- Self-contained Plotly HTML template (same pattern as duckdb_benchmarks) ---
# PLOTLY_TEMPLATE = """<!DOCTYPE html>
# <html>
# <head>
# <meta charset="utf-8">
# <script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
# <style>
#   body {{ margin: 0; padding: 0; background: #fff; overflow: visible; }}
#   #chart {{ width: 100vw; height: 100vh; overflow: visible; }}
#   .main-svg, .main-svg .draglayer, .main-svg .layer-above {{ overflow: visible !important; }}
#   svg {{ overflow: visible !important; }}
# </style>
# </head>
# <body>
# <div id="chart"></div>
# <script>
# var traces = {traces};
# var layout = {layout};
# var config = {{responsive: true, displayModeBar: false}};
# Plotly.newPlot('chart', traces, layout, config);
# window.addEventListener('resize', function() {{
#   Plotly.Plots.resize(document.getElementById('chart'));
# }});
# </script>
# </body>
# </html>"""

# AQI_BANDS = [
#     (0, 49, "rgba(0,128,0,0.2)", "Good"),
#     (50, 99, "rgba(255,255,0,0.2)", "Moderate"),
#     (100, 149, "rgba(255,165,0,0.2)", "Unhealthy for Some"),
#     (150, 199, "rgba(255,0,0,0.2)", "Unhealthy"),
#     (200, 299, "rgba(128,0,128,0.2)", "Very Unhealthy"),
#     (300, 500, "rgba(139,0,0,0.2)", "Hazardous"),
# ]
# AQI_LEGEND_COLORS = ["green", "yellow", "orange", "red", "purple", "darkred"]

# def make_aqi_shapes():
#     return [dict(type="rect", xref="paper", x0=0, x1=1, yref="y",
#                  y0=lo, y1=hi, fillcolor=c, line_width=0)
#             for lo, hi, c, _ in AQI_BANDS]

# def make_aqi_legend_traces():
#     return [dict(type="scatter", x=[None], y=[None], mode="markers",
#                  marker=dict(size=10, color=AQI_LEGEND_COLORS[i]),
#                  name=f"{label}: {lo}-{hi}", showlegend=True)
#             for i, (lo, hi, _, label) in enumerate(AQI_BANDS)]

# def make_layout(title):
#     return dict(
#         title=dict(text=title, font=dict(size=18)),
#         xaxis=dict(title="Date"),
#         yaxis=dict(title="PM2.5", type="log",
#                    tickvals=[10, 25, 50, 100, 250, 500],
#                    range=[0, 2.7]),
#         showlegend=True,
#         legend=dict(orientation="h", x=0.5, y=-0.25, xanchor="center",
#                     yanchor="top", font=dict(size=10),
#                     bgcolor="rgba(255,255,255,0.9)",
#                     bordercolor="#ccc", borderwidth=1),
#         margin=dict(t=60, b=140, l=60, r=30),
#         shapes=make_aqi_shapes(),
#     )

# def write_chart_html(filename, traces, title):
#     path = f"{CHART_DIR}/{filename}"
#     with open(path, "w") as f:
#         f.write(PLOTLY_TEMPLATE.format(
#             traces=json_mod.dumps(traces), layout=json_mod.dumps(make_layout(title))))
#     print(f"  Wrote {filename}")

# def run_hops(cmd):
#     result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
#     if result.returncode != 0:
#         print(f"Error: {cmd}\n{result.stderr}")
#     return result.stdout.strip()

# def hops_json(args):
#     result = subprocess.run(["hops"] + args + ["--json"], capture_output=True, text=True)
#     if result.returncode != 0:
#         return []
#     try:
#         return json_mod.loads(result.stdout)
#     except json_mod.JSONDecodeError:
#         return []

# def parse_id(output):
#     for word in output.split():
#         cleaned = word.strip("()")
#         if cleaned.isdigit():
#             return int(cleaned)
#     return None

# # --- Chart 1: Forecast (uses batch_data from earlier cells) ---
# forecast_dates = [str(d) for d in pd.to_datetime(batch_data['date']).dt.date]
# forecast_vals = [round(float(v), 2) for v in batch_data['predicted_pm25']]
# traces = [dict(type="scatter", mode="lines+markers", name="Predicted PM2.5",
#                x=forecast_dates, y=forecast_vals,
#                line=dict(color="red", width=2), marker=dict(size=6, color="blue"))]
# traces += make_aqi_legend_traces()
# write_chart_html("pm25_forecast.html", traces, f"PM2.5 Forecast — {city}, {street}")

# # --- Chart 2: Inference Hindcast (uses hindcast_df from earlier cells) ---
# hc_dates = [str(d) for d in pd.to_datetime(hindcast_df['date']).dt.date]
# hc_pred = [round(float(v), 2) for v in hindcast_df['predicted_pm25']]
# hc_actual = [round(float(v), 2) for v in hindcast_df['pm25']]
# traces = [
#     dict(type="scatter", mode="lines+markers", name="Predicted PM2.5",
#          x=hc_dates, y=hc_pred, line=dict(color="red", width=2),
#          marker=dict(size=6, color="blue")),
#     dict(type="scatter", mode="lines+markers", name="Actual PM2.5",
#          x=hc_dates, y=hc_actual, line=dict(color="black", width=2),
#          marker=dict(size=6, color="grey", symbol="triangle-up")),
# ]
# traces += make_aqi_legend_traces()
# write_chart_html("pm25_inference_hindcast.html", traces,
#                  f"PM2.5 Inference Hindcast — {city}, {street}")

# # --- Chart 3: Recent Air Quality History (uses air_quality_df from earlier cells) ---
# sorted_aq = air_quality_df.sort_values('date').tail(100)
# aq_dates = [str(d) for d in pd.to_datetime(sorted_aq['date']).dt.date]
# aq_vals = [round(float(v), 2) for v in sorted_aq['pm25']]
# traces = [dict(type="scatter", mode="lines+markers", name="Actual PM2.5",
#                x=aq_dates, y=aq_vals, line=dict(color="black", width=2),
#                marker=dict(size=4, color="grey"))]
# traces += make_aqi_legend_traces()
# write_chart_html("pm25_history.html", traces, f"PM2.5 Recent History — {city}, {street}")

# # --- Delete existing dashboard and charts (idempotent) ---
# existing_dashboards = hops_json(["dashboard", "list"])
# if existing_dashboards and isinstance(existing_dashboards, list):
#     for d in existing_dashboards:
#         if d.get("name") == "Air Quality Dashboard":
#             run_hops(f'hops dashboard delete {d["id"]}')
#             print(f"  Deleted dashboard {d['id']}")

# existing_charts = hops_json(["chart", "list"])
# if existing_charts and isinstance(existing_charts, list):
#     for c in existing_charts:
#         if "PM2.5" in c.get("title", ""):
#             run_hops(f'hops chart delete {c["id"]}')
#             print(f"  Deleted chart {c['id']}: {c['title']}")

# # --- Create charts and dashboard ---
# chart_configs = [
#     ("pm25_forecast.html", f"PM2.5 Forecast - {city}",
#      f"Air quality PM2.5 forecast for {city}, {street}"),
#     ("pm25_inference_hindcast.html", f"PM2.5 Inference Hindcast - {city}",
#      f"1-day-ahead PM2.5 predictions vs actuals for {city}, {street}"),
#     ("pm25_history.html", f"PM2.5 History - {city}",
#      f"Recent PM2.5 measurements for {city}, {street}"),
# ]

# chart_ids = []
# for filename, title, description in chart_configs:
#     url = f"Resources/charts/airquality/{filename}"
#     output = run_hops(f'hops chart create "{title}" --url "{url}" --description "{description}"')
#     cid = parse_id(output)
#     chart_ids.append(cid)
#     print(f"  Created chart '{title}' (ID: {cid})")

# output = run_hops('hops dashboard create "Air Quality Dashboard"')
# dash_id = parse_id(output)
# for i, cid in enumerate(chart_ids):
#     run_hops(f"hops dashboard add-chart {dash_id} --chart-id {cid} --width 24 --height 10 --x 0 --y {i * 10}")

# print(run_hops(f"hops dashboard info {dash_id}"))
# print(f"\nView dashboard at: {proj_url}/p/{project_id}/dashboards/{dash_id}")

---